In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 8221a972-5ccb-4783-8ca8-3c1cc198abd3
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 8221a972-5ccb-4783-8ca8-3c1cc198abd3 to get into ready status...
Session 8221a972-5ccb-4783-8ca8-3c1cc198abd3 

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("test").getOrCreate()

In [8]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [9]:
emp_schema = StructType([StructField('emp_id',IntegerType(),False),
                        StructField('emp_name',StringType(),False),
                        StructField('city',StringType(),False),
                        StructField('age',StringType(),False),
                        StructField('dob',DateType(),False),
                        StructField('department',StringType(),False),
                        StructField('salary',IntegerType(),False),
                        StructField('hire_date',DateType(),False),
                        StructField('gender',StringType(),False)])

In [14]:
df = spark.read.format('csv')\
.option('header','true')\
.option("inferSchema",'true')\
.option('mode','PERMISSIVE')\
.option('sep',',')\
.load("s3://pipeline-using-glue-automation/raw/")

In [15]:
df.show()

+------+--------------+---------+---+----------+----------+--------+----------+------+
|emp_id|      emp_name|     city|age|       dob|department|  salary| hire_date|gender|
+------+--------------+---------+---+----------+----------+--------+----------+------+
|     1|    Sneha Iyer|     NULL| 52|1984/05/07|     Sales|124161.0|2024/12/02|  Male|
|     2|   Rohit Gupta|   Mumbai| 44|1983/09/06|        IT| 38268.0|2020/10/06|Female|
|     3|Karan Malhotra|  Chennai| 28|1998/05/27|     Sales| 56965.0|2020/11/14|Female|
|     4|  Vikram Singh|     Pune| 23|1982/05/25| Marketing| 80747.0|2020/10/15|  Male|
|     5|   Divya Joshi|  Chennai| 57|1975/04/28|     Sales| 70949.0|2016/04/11|Female|
|     6|   Neha Kapoor|   Jaipur| 56|1999/07/25|        IT|110873.0|2016/05/30|Female|
|     7|  Suresh Kumar|   Mumbai| 32|1995/05/26|   Finance|    NULL|2016/01/06|Female|
|     8|   Arjun Reddy|    Delhi| 58|1978/10/10| Marketing|147206.0|2024/10/17|  Male|
|     9|    Sneha Iyer|   Jaipur| 35|1997/0

In [ ]:
#handle null values and remove duplicate data 

In [21]:
def cleansing_data():
    df = spark.read.format('csv')\
    .option('header','true')\
    .option("inferSchema",'true')\
    .option('mode','PERMISSIVE')\
    .option('sep',',')\
    .load("s3://pipeline-using-glue-automation/raw/")
    #handling null values in dataframe
    df = df.fillna({
    "salary": 0,
    "city": "Unknown",
    "department": "Not Assigned"
    })
    #handling duplicate data
    df = df.dropDuplicates()
    return df
    

In [25]:
df = cleansing_data()
df.count()


180


In [27]:
def save_clean_data_in_parquet_format_in_bronze(df):
    df.write.format('parquet').option("header",True).mode('overwrite').save('s3://pipeline-using-glue-automation/bronze/')
    print("data saved to bronze bucket folder")

In [28]:
df = cleansing_data()
save_clean_data_in_parquet_format_in_bronze(df)


data saved to bronze bucket folder
